# Grounded rhetorical-operation realization: inference only
NEW PRIVATE notebook. Attach a PRIVATE Dataset containing ONLY operation_pack.json, plus the outputs of oliverlds/plan_realizer_pilot version 1 (must include adapter_planned/adapter_config.json and adapter_model.safetensors). Enable internet, GPU T4 x2 and HF_TOKEN in Kaggle Secrets. Only GPU 0 is used. No credentials in files.
30 first-attempt greedy calls: base Gemma 3 4B and the frozen planned adapter, each under ordinary, shallow count/length plan and grounded-operation conditions on the same five paragraphs. The four transformation cases and one already-parallel preservation control are reported separately. ALL FIVE WERE TRAINING EXAMPLES SEEN BY THE ADAPTER. No development/test rows, new training, adapter updates or learned mapping.
All conditions share byte-identical source claim evidence, supplemental facts and existing relationship evidence. Ordinary therefore means no additional plan, with this common source scaffold. This differs from earlier pilot prompts; no exact historical generation reproduction is asserted. Only the plan guidance changes. The reference wording/annotations/cues are never passed to the generator. Plans are reference-informed oracle side information.
The saved adapter was trained on shallow plans, not these new operations or the shared evidence scaffold. Its operation response is an out-of-format realization diagnostic, not evaluation of an operation-trained adapter or generalization. Adapter files and loaded tensor hashes are checked against the reviewed pilot. No optimizer or backward pass exists here.
Pinned base/NLI revisions, NF4/FP32 on one T4, 512 output-token cap and 4096 total context limit. Audit all rendered prompts before loading weights; never truncate. Empty/cutoff responses stay failures and remaining calls continue; format issues remain verbatim. No retries, repairs or reference fallback. Gemma unloads before NLI.
Save full semantic verdicts and claim-to-output-sentence NLI matrices. Location hypotheses cannot prove operation adherence, resolve every composite claim, or establish author style. Assess claim drift, meaningful style change, copying and rhetorical realization separately, with the existing allowance for minor drift. A preservation-control match is not a style success.
Download /kaggle/working/grounded_operation_realization_v1/: raw generations, jobs, metrics, full verifier/localization evidence, evaluation, token audit, adapter identity, runtime, run status and manifest. No GPU run was submitted by the builder.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"


In [ ]:
%pip install -q "transformers==5.20.1" "peft==0.21.2" "accelerate==1.14.0" "bitsandbytes==0.50.2" "huggingface-hub==1.33.0" "sentencepiece>=0.2"


In [ ]:
import base64, json, sys, zlib
from pathlib import Path
runtime = Path("/kaggle/working/_operation_runtime")
files = json.loads(zlib.decompress(base64.b64decode('...')).decode("utf-8"))
for name, source in files.items():
    path = runtime / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
sys.path.insert(0, str(runtime / "src"))


In [ ]:
import importlib
experiment = importlib.import_module("fama_style.rhetoric.operation_realization")
matches = list(Path("/kaggle/input").rglob("operation_pack.json"))
if len(matches) != 1:
    raise ValueError("Attach exactly one private operation_pack.json")
arguments = ["--pack", str(matches[0]), "--expected-pack-sha", '7f475a2e872343a4a614da8da60d3c6efa1bd1a8ca3b19f38a0392abed3ab9a3', "--adapter-root", "/kaggle/input"]
experiment.main(arguments + ["--output-dir", "/kaggle/working/operation_preflight"])


In [ ]:
experiment.main(arguments + ["--execute-inference", "--output-dir", "/kaggle/working/grounded_operation_realization_v1"])
